# B2 — Classification end-to-end

**From:** "predicting numbers" (B1)  **To:** predicting *categories* with probabilities, the loss LLMs actually train on, and the overfitting trap plus its fix.

Our project asks categorical questions constantly: task completed — yes or no? Call acceptable — yes or no? A regression line outputs any number; we need **"yes with probability 0.83."** Three new ideas chain together: **logit → sigmoid → cross-entropy.**

The toy world below mimics our calls: two features per call (median response gap, number of overlaps), and a truth label "failed call?" — synthetic, but shaped like the real thing.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "rubric.yaml").exists())
import sys
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
print("ready · repo:", ROOT.name)

n = 120
good = np.column_stack([rng.normal(550, 150, n // 2), rng.normal(1.6, 1.0, n // 2)])
bad  = np.column_stack([rng.normal(820, 210, n // 2), rng.normal(2.8, 1.4, n // 2)])
X = np.vstack([good, bad])
y = np.array([0] * (n // 2) + [1] * (n // 2))
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.scatter(X[y == 0, 0], X[y == 0, 1], label="ok call", alpha=0.7)
ax.scatter(X[y == 1, 0], X[y == 1, 1], label="failed call", alpha=0.7, marker="x")
ax.set_xlabel("median gap (ms)"); ax.set_ylabel("overlap count"); ax.legend()
ax.set_title("synthetic calls in feature space"); plt.show()

## Logits and the squash
A linear model scores each point: **z = w₁·gap + w₂·overlaps + b**. That raw score is called a **logit** — any real number, more positive = more "failed-ish." To bet money we need a probability in (0,1). The **sigmoid** squashes: σ(z) = 1/(1+e^(−z)). (Its n-class sibling, **softmax**, does the same for many categories — that is the last layer of every LLM, turning logits over ~100k tokens into next-token probabilities.)

**PREDICT:** what does scaling all logits by 5 (z → 5z) do to the sigmoid curve — and to the model's *confidence*?

In [ ]:
z = np.linspace(-6, 6, 200)
sig = lambda z: 1 / (1 + np.exp(-z))
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(z, sig(z), label="sigmoid(z)")
ax.plot(z, sig(5 * z), label="sigmoid(5z) - sharper")
ax.set_xlabel("logit z"); ax.set_ylabel("probability"); ax.legend()
ax.set_title("the squash; scaling logits = confidence knob"); plt.show()

Scaling logits sharpens confidence without changing *which* side wins. File that away: **temperature in LLMs is exactly this knob in reverse** (divide logits by T; T→0 sharpens toward certainty). Book C1 picks it up.

## Cross-entropy: the loss of "surprise"
For a true label, the loss is **−log(probability the model gave the truth)**. Confidently right → tiny loss. Unsure → moderate. **Confidently wrong → enormous.** This asymmetry is the point: it brutalizes overconfident error.

In [ ]:
for p_truth, story in [(0.95, "confidently RIGHT"), (0.5, "shrugging"), (0.05, "confidently WRONG")]:
    print(f"model gave truth p={p_truth:4.2f} ({story:18s}) -> loss {-np.log(p_truth):5.2f}")

LLMs train on exactly this loss, token by token: "the truth was token X — how surprised were you?" Trillions of small surprises, descended by gradient. You now know the loss function of the entire modern era.

## Train it (B1's loop, three parameters now)

In [ ]:
Xs = (X - X.mean(0)) / X.std(0)            # standardize features so one lr fits both
w, b, lr = np.zeros(2), 0.0, 0.5
for step in range(400):
    p = sig(Xs @ w + b)
    grad_w = Xs.T @ (p - y) / len(y)       # gradients of cross-entropy (take on faith today)
    grad_b = (p - y).mean()
    w -= lr * grad_w; b -= lr * grad_b
acc = ((sig(Xs @ w + b) > 0.5) == y).mean()
print(f"train accuracy: {acc:.0%}   weights: gap {w[0]:+.2f}, overlaps {w[1]:+.2f}")

gx, gy = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))
pp = sig(np.column_stack([gx.ravel(), gy.ravel()]) @ w + b).reshape(gx.shape)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.contourf(gx, gy, pp, levels=20, cmap="RdBu_r", alpha=0.65)
ax.scatter(Xs[y == 0, 0], Xs[y == 0, 1], label="ok")
ax.scatter(Xs[y == 1, 0], Xs[y == 1, 1], marker="x", label="failed")
ax.set_xlabel("gap (standardized)"); ax.set_ylabel("overlaps (standardized)"); ax.legend()
ax.set_title("decision field: shading = model's P(failed)"); plt.show()

(How to read a **contour/decision plot**: the background shading is the model's probability at every possible point — deep blue "surely ok," deep red "surely failed," the pale band between them is the **decision boundary**, where the model genuinely does not know.)

Both weights came out positive — bigger gaps and more overlaps push toward "failed." The model *discovered* the rubric's direction from data.

## Overfitting: the trap, seen with eyes
A model can ace training data by **memorizing** it. The honest test is data it never saw. Watch a memorizer (1-nearest-neighbor: copy the label of the closest training point) vs our humble line, on fresh validation calls:

In [ ]:
def fresh(n):
    g = np.column_stack([rng.normal(550, 150, n // 2), rng.normal(1.6, 1.0, n // 2)])
    b = np.column_stack([rng.normal(820, 210, n // 2), rng.normal(2.8, 1.4, n // 2)])
    Xv = (np.vstack([g, b]) - X.mean(0)) / X.std(0)
    return Xv, np.array([0] * (n // 2) + [1] * (n // 2))

def knn1(Xq):
    d = ((Xq[:, None, :] - Xs[None, :, :]) ** 2).sum(-1)
    return y[d.argmin(1)]

Xv, yv = fresh(200)
rows = [("memorizer (1-NN)", (knn1(Xs) == y).mean(), (knn1(Xv) == yv).mean()),
        ("logistic line",    ((sig(Xs@w+b) > .5) == y).mean(), ((sig(Xv@w+b) > .5) == yv).mean())]
print(f"{'model':<18} {'train acc':>9} {'VAL acc':>9}")
for name, tr, va in rows:
    print(f"{name:<18} {tr:>9.0%} {va:>9.0%}")

The memorizer is perfect on what it saw and *worse* on what it did not — that spread is the **generalization gap**, ML's lie detector. The humble line, which could not memorize, holds steady. Moral: **always score on held-out data**, and distrust any perfect training number.

Bridge to our project: this is *the same epistemology* as judging the judge with **blind** human labels (F3) — performance claimed on data the system could fit is not evidence.

## Self-check
1. Logit, sigmoid, probability — chain them in one sentence.
2. Why is cross-entropy brutal specifically to confident error, and why is that desirable?
3. What is the generalization gap and what does a large one scream?
4. Connect temperature to today's "scale the logits" demo.
5. **Gotcha:** your protein model reports 99.4% training accuracy. Your first question?

<details><summary>Answers</summary>

1. The model emits a raw score (logit); sigmoid squashes it into (0,1); that squashed value is the claimed probability.
2. Loss −log p explodes as p(truth)→0; a model that bets hard and wrong must be punished harder than one that shrugs, or it learns to bluff.
3. Train-vs-heldout performance spread; a big gap screams memorization, not learning.
4. Temperature divides logits before softmax: T→0 sharpens toward argmax certainty (our judge's setting), T high flattens toward dice.
5. "And on data it has never seen?" — no validation number, no conversation.
</details>